## Routers (endpoints)
En este programa, se realizan las rutas que se utilizarán para hacer los llamados a las funciones, etc.

### Integrantes:

Arana Acosta Shantal Vanessa

Cortés Reyes Karla Fátima 

Domínguez Jiménez Ana Andrea 

Higuera Pineda Ángel Abraham 

In [27]:
#Parte del codigo, para una correcta ejecucion

import sys
import os

#Subimos DOS niveles: de Analisis_IA/ → raíz del proyecto
ruta_proyecto = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))

#Apuntamos a Programa_sinnotebook donde están los .py reales
ruta_base = os.path.join(ruta_proyecto, "Programa_sinnotebook")

if ruta_base not in sys.path:
    sys.path.insert(0, ruta_base)

print("Ruta base añadida:", ruta_base)
print("Existe:", os.path.exists(ruta_base))

Ruta base añadida: /home/gamesrack/Documentos/Quinto_Semestre/Codigos/Python/Lenguaje_natural/Proyecto_spam_git/Proyecto-PLN-Detector-de-SPAM-SMS-mexicano/Programa_sinnotebook
Existe: True


In [28]:
#Librerias:
#Fastapi: Framework para construir APIs de manera rápida y sencilla.
from fastapi import APIRouter
#Typing: Para anotaciones de tipos, como List.
from typing import List
#Sys y os: Para manejo de rutas y archivos
import sys
import os

from Analisis_IA import schemas
from Analisis_IA import analyser
from Modelo_NLP import detector

In [29]:
#Creamos el router para definir las rutas de la API
router = APIRouter()

In [30]:
#Routers para la API

# Endpoint para verificar que el servicio está activo y funcionando correctamente
@router.get("/activo")
async def check_activo():
    return {"status": "activo"}


# Endpoint principal para procesar los mensajes SMS recibidos desde Android.
@router.post("/clasificar-sms", response_model=List[schemas.GrupoSpamOutput])
async def procesar_mensajes_sms(mensajes: List[schemas.SmsInput]):
    """
    Flujo Real: 
    1. Recibe todos los SMS.
    2. Pasa por el modelo SVM local para filtrar solo el SPAM.
    3. Manda el SPAM a Gemini para agruparlo y resumirlo.
    """
    if not mensajes:
        return []

    #1. Convertimos los modelos de Pydantic a diccionarios
    mensajes_crudos = [msg.model_dump() for msg in mensajes]
    
    #2. EL CEREBRO: Filtramos usando el modelo SVM
    spam_detectado = detector.detectar_spam(mensajes_crudos)
    
    #Si el SVM dice que todo está limpio (no hay spam), regresamos una lista vacía
    if not spam_detectado:
        return []

    #3. EL ASISTENTE: Mandamos los mensajes maliciosos a Gemini para generar el resumen final
    spam_resumido = analyser.resumir_spam_detectado(spam_detectado)
    
    return spam_resumido